# Improved Startup Data Preprocessing


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Data


In [ ]:
companies = pd.read_csv("../data/companies.csv", encoding="latin1", low_memory=False)
investments = pd.read_csv("../data/investments_VC.csv", encoding="latin1", low_memory=False)
acquisitions = pd.read_csv("../data/acquisitions.csv", encoding="latin1", low_memory=False)

## 2. Basic Cleaning & Deduplication


In [ ]:
companies.drop_duplicates(inplace=True)
investments.drop_duplicates(inplace=True)
acquisitions.drop_duplicates(inplace=True)

companies = companies[[
    "name",
    "category_code",
    "country_code",
    "status",
    "permalink",
    "short_description",
    "description",
    "overview",
    "invested_companies",
    "funding_total_usd",
    "funding_rounds"
]]

## 3. Handle Missing Values


In [ ]:
# Drop rows where unique identifiers are missing
companies.dropna(subset=["name", "permalink"], inplace=True)

# Clean and convert numerical columns
def clean_currency(x):
    if isinstance(x, str):
        x = x.replace(',', '').replace('-', '')
    try:
        return float(x)
    except ValueError:
        return np.nan

companies["funding_total_usd"] = companies["funding_total_usd"].apply(clean_currency)
companies["funding_rounds"] = pd.to_numeric(companies["funding_rounds"], errors='coerce')
companies["invested_companies"] = pd.to_numeric(companies["invested_companies"], errors='coerce')

# Fill missing numerical values with 0
companies["funding_total_usd"] = companies["funding_total_usd"].fillna(0)
companies["funding_rounds"] = companies["funding_rounds"].fillna(0)
companies["invested_companies"] = companies["invested_companies"].fillna(0)

# Fill categorical text columns with 'Unknown'
for col in ["category_code", "country_code", "status"]:
    companies[col] = companies[col].fillna("Unknown")

# Fill description columns with empty strings
for col in ["short_description", "description", "overview"]:
    companies[col] = companies[col].fillna("")

## 4. Text Synthesis


In [ ]:
# Typecasting
for col in ["name", "category_code", "country_code", "status", "permalink"]:
    companies[col] = companies[col].astype(str)

def get_description(row):
    if row["description"].strip():
        return row["description"]
    elif row["overview"].strip():
        return row["overview"]
    elif row["short_description"].strip():
        return row["short_description"]
    else:
        return "No detailed description available"

def build_text(row):
    description = get_description(row)
    funding = f"{row['funding_total_usd']:,.0f}" if row['funding_total_usd'] > 0 else "Unknown"
    
    return f'''
    Startup: {row['name']}.
    Description: {description}.
    Industry: {row['category_code']}.
    Country: {row['country_code']}.
    Funding: {funding} USD.
    Funding rounds: {int(row['funding_rounds'])}.
    Status: {row['status']}.
    '''

companies["text"] = companies.apply(build_text, axis=1)

## 5. Final Output Generation


In [ ]:
companies = companies.reset_index(drop=True)
companies["id"] = companies.index.map(lambda x: f"startup-{x}")

companies = companies.rename(columns={
    "category_code": "industry",
    "country_code": "country",
})

# Keep permalink to allow future joins with investments/acquisitions
final_df = companies[["id", "permalink", "name", "text", "industry", "country"]]
final_df.head()

,id,permalink,name,text,industry,country
0,startup-0,/company/wetpaint,Wetpaint,\n Startup: Wetpaint.\n Description: Tec...,web,USA
1,startup-1,/company/flektor,Flektor,\n Startup: Flektor.\n Description: Flek...,games_video,USA
2,startup-2,/company/there,There,\n Startup: There.\n Description: There....,games_video,USA
3,startup-3,/company/mywebbo,MYWEBBO,\n Startup: MYWEBBO.\n Description: BRAN...,network_hosting,Unknown
4,startup-4,/company/the-movie-streamer,THE Movie Streamer,\n Startup: THE Movie Streamer.\n Descri...,games_video,Unknown


In [ ]:
output_path = "../data/startups_data.csv"
final_df.to_csv(output_path, index=False)
print(f"Successfully saved {len(final_df)} records to {output_path}")

Successfully saved 196530 records to ../data/startups_data.csv
